# Per-cell Q-trajectories + running-EMA (mock of weight averaging)

For runs logged with `--log-q-grid`. The key view: each action's **raw Q (faint, oscillating)** with
a **running EMA on top (bold)** — the EMA is computed *at every checkpoint*
(`ema_t = d·ema_{t-1} + (1-d)·q_t`), so the averaged estimate is itself a *trajectory* that settles,
not a single flat line. The EMA's final ordering is the "averaged" decision (the weight-EMA / SWA
sibling).

Sections: (1) trajectory grids; (2) cells the EMA **changes**, outlined green = correction / red =
mistake; (3) side-by-side policy heatmaps (raw snapshot vs EMA); (4) corrections/mistakes list.
Tune `EMA_DECAY` (higher = smoother / longer memory).

In [ ]:
import json, glob, os
from pathlib import Path
from math import ceil
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

RUNS = Path("..") / "runs"
EMA_DECAY = 0.90  # per-checkpoint EMA decay; raise toward 0.97 for smoother / longer memory
RECORDS = []
for f in sorted(RUNS.glob("*/record.json")):
    try:
        r = json.load(open(f)); r["_id"] = f.parent.name; RECORDS.append(r)
    except Exception as e:
        print("skip", f, e)
is_dqn = lambda r: r.get("method") == "dqn"
have_grid = [r for r in RECORDS if is_dqn(r) and any("q_grid" in p for p in r.get("learning_curve", []))]
cell_label = lambda c: f"{'soft' if c['is_soft'] else 'hard'}{c['player_value']}_v{c['dealer_upcard']}"
ACT_COLOR = {"hit": "tab:blue", "stand": "tab:orange", "double": "tab:green", "split": "tab:red"}
ACT1 = {"hit": "H", "stand": "S", "double": "D", "split": "P", "surrender": "R"}

def ema_series(xs, decay=EMA_DECAY):
    out, e = [], None
    for x in xs:
        e = x if e is None else decay * e + (1 - decay) * x
        out.append(e)
    return out

def series(r):
    gp = [p for p in r.get("learning_curve", []) if "q_grid" in p]
    if not gp:
        return None
    eps = [p["episode"] for p in gp]
    labels = list(gp[-1]["q_grid"].keys())
    actions = list(gp[-1]["q_grid"][labels[0]].keys())
    out = {}
    for l in labels:
        d = {}
        for a in actions:
            raw = [p["q_grid"].get(l, {}).get(a) for p in gp]
            d[a] = (raw, ema_series(raw))
        out[l] = d
    return eps, labels, actions, out

def policies(r):
    res = series(r)
    if res is None:
        return None
    eps, labels, actions, out = res
    basic = {cell_label(c): c["basic_action"] for c in r["diff"]["cells"]}
    info = {cell_label(c): (c["player_value"], c["is_soft"], c["dealer_upcard"]) for c in r["diff"]["cells"]}
    labels = [l for l in basic if l in out]
    snap = {l: max(actions, key=lambda a: out[l][a][0][-1]) for l in labels}   # raw final
    ema = {l: max(actions, key=lambda a: out[l][a][1][-1]) for l in labels}    # EMA final
    return eps, actions, out, basic, info, labels, snap, ema

def outline(ax, color, lw=2.6):
    for sp in ax.spines.values():
        sp.set_color(color); sp.set_linewidth(lw)

print(f"{len(have_grid)} runs have q_grid:", [r['_id'] for r in have_grid], "| EMA_DECAY =", EMA_DECAY)

## 1. Trajectory grids — faint = raw Q, bold = running EMA

In [ ]:
def plot_traj(r, cols=5, max_cells=60):
    res = policies(r)
    if res is None:
        print("no q_grid"); return
    eps, actions, out, basic, info, labels, snap, ema = res
    dis = [cell_label(c) for c in r["diff"]["cells"] if c["category"] == "genuine_disagreement"][:max_cells]
    n = len(dis); rows = ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 2.4 * rows), sharex=True)
    axes = np.array(axes).reshape(-1)
    for ax, l in zip(axes, dis):
        for a in actions:
            raw, em = out[l][a]
            ax.plot(eps, raw, color=ACT_COLOR.get(a, "gray"), lw=0.7, alpha=0.25)
            ax.plot(eps, em, color=ACT_COLOR.get(a, "gray"), lw=1.9, alpha=0.95, label=a)
        mk = "OK" if ema[l] == basic[l] else "X"
        ax.set_title(f"{l} raw={snap[l][0].upper()} ema={ema[l][0].upper()}[{mk}] bas={basic[l][0].upper()}", fontsize=7)
        ax.grid(alpha=.3); ax.tick_params(labelsize=6); ax.axhline(0, color="k", lw=.4, alpha=.4)
    for ax in axes[n:]:
        ax.axis("off")
    axes[0].legend(fontsize=7)
    fig.suptitle(f"{r['_id']}  (bold = EMA d={EMA_DECAY})  snapshot agree={r['diff']['agreement_unweighted']*100:.1f}%", y=1.004)
    plt.tight_layout(); plt.show()

for r in have_grid:
    plot_traj(r)

## 2. Cells the EMA changes — green outline = correction, red = mistake

In [ ]:
def plot_changes(r, cols=5):
    res = policies(r)
    if res is None:
        print("no q_grid"); return
    eps, actions, out, basic, info, labels, snap, ema = res
    fixed = [l for l in labels if snap[l] != basic[l] and ema[l] == basic[l]]
    broke = [l for l in labels if snap[l] == basic[l] and ema[l] != basic[l]]
    changed = [(l, "green") for l in fixed] + [(l, "red") for l in broke]
    if not changed:
        print(f"{r['_id']}: EMA changed nothing"); return
    n = len(changed); rows = ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 2.5 * rows), sharex=True)
    axes = np.array(axes).reshape(-1)
    for ax, (l, col) in zip(axes, changed):
        for a in actions:
            raw, em = out[l][a]
            ax.plot(eps, raw, color=ACT_COLOR.get(a, "gray"), lw=0.7, alpha=0.25)
            ax.plot(eps, em, color=ACT_COLOR.get(a, "gray"), lw=1.9, alpha=0.95, label=a)
        kind = "correction" if col == "green" else "MISTAKE"
        ax.set_title(f"{l}  {snap[l][0].upper()}->{ema[l][0].upper()} (bas {basic[l][0].upper()})  {kind}", fontsize=7, color=col)
        ax.grid(alpha=.3); ax.tick_params(labelsize=6); ax.axhline(0, color="k", lw=.4, alpha=.4); outline(ax, col)
    for ax in axes[n:]:
        ax.axis("off")
    axes[0].legend(fontsize=7)
    fig.suptitle(f"EMA changes — {r['_id']}: {len(fixed)} corrections (green) / {len(broke)} mistakes (red)", y=1.004, fontsize=12)
    plt.tight_layout(); plt.show()

for r in have_grid:
    plot_changes(r)

## 3. Side-by-side policy heatmaps — raw snapshot vs EMA (green agree / red disagree)

In [ ]:
def heatmaps(r):
    res = policies(r)
    if res is None:
        print("no q_grid"); return
    eps, actions, out, basic, info, labels, snap, ema = res
    cmap = ListedColormap(["#c8e6c9", "#ef9a9a"])
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    for row, soft in enumerate([False, True]):
        vals = sorted({info[l][0] for l in labels if info[l][1] == soft}); ups = list(range(2, 12))
        sub = [l for l in labels if info[l][1] == soft]
        for col, (name, pol) in enumerate([("raw snapshot", snap), ("EMA", ema)]):
            ax = axes[row][col]; grid = np.full((len(vals), len(ups)), np.nan)
            for l in sub:
                v, s, u = info[l]; i = vals.index(v); j = ups.index(u)
                grid[i, j] = 0 if pol[l] == basic[l] else 1
                ax.text(j, i, ACT1[pol[l]], ha="center", va="center", fontsize=7)
            ax.imshow(grid, cmap=cmap, vmin=0, vmax=1, aspect="auto")
            ax.set_xticks(range(len(ups))); ax.set_xticklabels(ups); ax.set_yticks(range(len(vals))); ax.set_yticklabels(vals)
            agr = sum(pol[l] == basic[l] for l in sub) / len(sub) * 100
            ax.set_title(f"{'soft' if soft else 'hard'} — {name}  ({agr:.0f}% agree)", fontsize=10)
            ax.set_xlabel("dealer upcard"); ax.set_ylabel("player total")
    plt.suptitle(f"Policy vs basic — raw vs EMA   ({r['_id']})", fontsize=12)
    plt.tight_layout(); plt.show()

for r in have_grid:
    heatmaps(r)

## 4. Corrections vs mistakes (list)

In [ ]:
def compare(r):
    res = policies(r)
    if res is None:
        print(f"{r['_id']}: no q_grid"); return
    eps, actions, out, basic, info, labels, snap, ema = res
    fixed = [l for l in labels if snap[l] != basic[l] and ema[l] == basic[l]]
    broke = [l for l in labels if snap[l] == basic[l] and ema[l] != basic[l]]
    sa = sum(snap[l] == basic[l] for l in labels) / len(labels) * 100
    ea = sum(ema[l] == basic[l] for l in labels) / len(labels) * 100
    print(f"=== {r['_id']} (enc={r['config'].get('encoding')} seed={r['config'].get('seed')}) ===")
    print(f"raw snapshot {sa:.1f}%  ->  EMA {ea:.1f}%   (net {ea-sa:+.1f})")
    print(f"  CORRECTIONS: {len(fixed)}")
    for l in fixed:
        print(f"     + {l:14} {snap[l]:6} -> {ema[l]:6}  (basic {basic[l]})")
    print(f"  MISTAKES: {len(broke)}")
    for l in broke:
        print(f"     - {l:14} {snap[l]:6} -> {ema[l]:6}  (basic {basic[l]})")
    print()

for r in have_grid:
    compare(r)